In [5]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import skimage as ski 
from skimage import exposure 
from skimage.feature import peak_local_max 
from skimage.filters import threshold_multiotsu 
from skimage.measure import regionprops 
from skimage.segmentation import watershed 
from scipy import ndimage as ndi 
from miseval import evaluate 
from pathlib import Path 
from skimage import metrics 

segmentation_results = Path("segmentation_results") 
segmentation_results.mkdir(exist_ok=True) 

metrics_hausdorff = [] 
metrics_jaccard = [] 
metrics_dice = [] 
image_number = [] 

# Check file paths
microscopy_collection = ski.io.imread_collection("CHAT/images/*tif")
gt_col = ski.io.imread_collection("mask/masks/*png")

max_images = min(len(microscopy_collection), len(gt_col))

# CRITICAL PATH CHECK: Ensure images are actually loaded
print(f"Microscopy images loaded: {len(microscopy_collection)}")
print(f"Ground truth masks loaded: {len(gt_col)}")
print(f"Max matching images to process: {max_images}")

if max_images == 0:
    print("\n[WARNING]: No matching images found! Please check your file paths.")

rng = np.random.default_rng(seed=58) 
i = 0 
attempts = 0  # Safety tracker to prevent infinite loops
max_attempts = max_images * 3

while i <= 60 and i < max_images and attempts < max_attempts:
    attempts += 1
    image_index = rng.integers(0, max_images - 1) 
    if image_index in image_number: 
        continue 
        
    image = microscopy_collection[image_index] 
    smoothed = ski.filters.gaussian(image) 
    sharpened = ski.filters.unsharp_mask(smoothed, amount = 25) 
    contrast_enhancement = exposure.adjust_gamma(sharpened, 0.6) 
    threshold = threshold_multiotsu(contrast_enhancement, classes = 2 ) 
    region = np.digitize(contrast_enhancement, bins = threshold - 0.01) 
    foreground = region > 0 
    clean_objects = ski.morphology.remove_small_objects(foreground, max_size= 80) 
    noise_removal = ski.morphology.remove_small_holes(clean_objects , max_size= 70) 
    distance = ndi.distance_transform_edt(noise_removal) 
    coords = peak_local_max(distance, min_distance = 13, labels=noise_removal) 
    mask = np.zeros(distance.shape, dtype=bool) 
    mask[tuple(coords.T)] = True 
    markers, number_of_markers = ndi.label(mask) 
    labels = watershed(-distance, markers, mask=noise_removal,) 
    
    rng_colors = np.random.default_rng(seed=79) 
    number_of_labels = labels.max() 
    random_colors = rng_colors.random((number_of_labels + 1, 3)) 
    random_colors[0] = [0, 0, 0] 
    shuffled_colormask = random_colors[labels] 
    
    # Ground Truth------------------------------------------------ 
    gt_image = gt_col[image_index] 
    gt_image = gt_image[:,:,0] 
    gt_image = ski.morphology.label(gt_image) 
    ground = gt_image.astype(np.int32)
    
    # Corrected placement to avoid appending faulty metrics
    if np.count_nonzero(ground) == 0: 
        continue 
        
    y_true_flat = ground.flatten() 
    y_pred_flat = labels.flatten() 
    gt1 = (y_true_flat > 0).astype(int) 
    seg = (y_pred_flat > 0).astype(int) 
    removed_gt = ground > 0 
    removed_seg = labels > 0 
    
    average_hausdorff_distance = ski.metrics.hausdorff_distance(removed_gt, removed_seg, method = 'modified') 
    dice = evaluate(gt1, seg, metric='DSC') 
    jaccard = evaluate(gt1, seg, metric='IoU') 
    
    metrics_dice.append(dice) 
    metrics_jaccard.append(jaccard) 
    metrics_hausdorff.append(average_hausdorff_distance) 
    image_number.append(image_index) 
    
    if i < 6: 
        fig, ax = plt.subplots(ncols=3, figsize=(20, 20)) 
        ax[0].imshow(image, cmap ='gray') 
        ax[0].set_title('Original') 
        ax[1].imshow(shuffled_colormask) 
        ax[1].set_title('Segmentation prediction') 
        ax[2].imshow(ground, cmap = plt.cm.nipy_spectral) 
        ax[2].set_title('Ground Truth') 
        plt.savefig(segmentation_results / f"Segmentation_Example{i+1}.png", dpi=300) 
        plt.close() 
        
    i = i + 1 

# --- Calculations wrapped to prevent empty slice errors ---
if len(image_number) == 0:
    print("\n[ERROR]: Calculations aborted because zero images were successfully evaluated.")
else:
    AHD = np.array(metrics_hausdorff) 
    DSI = np.array(metrics_dice) 
    IoU = np.array(metrics_jaccard) 
    index = np.array(image_number) 
    
    MeanAverage_Hausdorff_Distance = np.mean(metrics_hausdorff) 
    Standard_Deviation_of_Hausdorff_Distance = np.std(metrics_hausdorff) 
    Average_Intersection_Over_Union = np.mean(metrics_jaccard) 
    Standard_Deviation_of_Intersection_Over_Union = np.std(metrics_jaccard) 
    Average_Dice_Similarity_Index = np.mean(metrics_dice) 
    Standard_Deviation_of_Dice_Similarity_Index = np.std(metrics_dice) 
    
    #----------------------- 
    df2 = pd.DataFrame({ 
        "Image Number": index, 
        "Average Hausdorff Distance": AHD, 
        "Dice Coefficient": DSI, 
        "Intersection over Union ": IoU, 
    }) 
    
    meanstd = pd.DataFrame({ 
        "" : pd.Categorical(["Average", "Standard Deviation"]), 
        "Average Hausdorff Distance": np.array([MeanAverage_Hausdorff_Distance, Standard_Deviation_of_Hausdorff_Distance]), 
        "Intersection over Union": np.array([Average_Intersection_Over_Union, Standard_Deviation_of_Intersection_Over_Union]), 
        "Dice Coefficient" : np.array([Average_Dice_Similarity_Index, Standard_Deviation_of_Dice_Similarity_Index]), 
    }) 
    
    df2.to_csv(segmentation_results / 'BBBC039_segmentation_evaluation.csv', index = False) 
    meanstd.to_csv(segmentation_results / 'BBBC039_SegmentationMetrics_mean,std.csv', index = False) 
    print("\n-------Evaluation Complete-----") 

print(f"Current Working Directory: {Path.cwd()}")


Microscopy images loaded: 200
Ground truth masks loaded: 200
Max matching images to process: 200

-------Evaluation Complete-----
Current Working Directory: C:\Users\Anant


In [ ]:
Project Description: calculate hausdorff distance, jaccard index, and DICE score vomparitively to the ground Truth:

Parameters:
use approximatey 40-50 images to prevent bias use images randomly generated 

Successful sgementation examples: 
16,131,11, 24 (w/ ground truth example,and original image) hausdorff < 1 
Higher than median hausdorff 177, 169 hausdorff > 1
Lower than average Jaccard: 178 

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage as ski
from skimage import exposure
from skimage.feature import peak_local_max
from skimage.filters import threshold_multiotsu
from skimage.measure import regionprops
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from miseval import evaluate
from pathlib import Path
from skimage import metrics


segmentation_results = Path("segmentation_results")
segmentation_results.mkdir(exist_ok=True)

metrics_hausdorff = []
metrics_jaccard = []
metrics_dice = []
image_number = []
microscopy_collection = ski.io.imread_collection("CHAT/images/*tif")# replace with custom file path of original microscopy images
gt_col = ski.io.imread_collection("mask/masks/*png")# replace with custom file path for ground truth masks 
max_images = min(len(microscopy_collection), len(gt_col))
rng = np.random.default_rng(seed=58) 

i = 0 
while i <= 60 and i < max_images:
    image_index = rng.integers(0, max_images - 1)
    if image_index in image_number:
        continue 
    
    image = microscopy_collection[image_index]
    smoothed = ski.filters.gaussian(image)
    sharpened = ski.filters.unsharp_mask(smoothed, amount = 25)
    contrast_enhancement = exposure.adjust_gamma(sharpened, 0.6)
    threshold = threshold_multiotsu(contrast_enhancement, classes = 2 )
    region = np.digitize(contrast_enhancement, bins = threshold - 0.01)
    foreground = region > 0
    clean_objects  = ski.morphology.remove_small_objects(foreground, max_size= 80) 
    noise_removal = ski.morphology.remove_small_holes(clean_objects , max_size= 70)
    distance = ndi.distance_transform_edt(noise_removal)
    coords = peak_local_max(distance, min_distance = 13, labels=noise_removal)
    mask = np.zeros(distance.shape, dtype=bool)
    mask[tuple(coords.T)] = True
    markers, number_of_markers = ndi.label(mask) 
    labels = watershed(-distance, markers, mask=noise_removal,) 
    rng = np.random.default_rng(seed=79)
    number_of_labels = labels.max()
    random_colors = rng.random((number_of_labels + 1, 3))
    random_colors[0] = [0, 0, 0]
    shuffled_colormask = random_colors[labels] #shuffled colormask for easier viewing 


    
    # Ground Truth------------------------------------------------
    gt_image = gt_col[image_index]
    gt_image = gt_image[:,:,0]
    gt_image = ski.morphology.label(gt_image)
    ground = gt_image.astype(np.int32)#final ground truth image 
    if np.count_nonzero(ground) == 0:
        continue
    y_true_flat = ground.flatten()
    y_pred_flat = labels.flatten()
    gt1 = (y_true_flat > 0).astype(int)
    seg = (y_pred_flat > 0).astype(int)
    removed_gt = ground >0 
    removed_seg = labels > 0
    average_hausdorff_distance = ski.metrics.hausdorff_distance(removed_gt, removed_seg, method = 'modified')
    dice = evaluate(gt1, seg, metric='DSC', )
    jaccard = evaluate(gt1, seg, metric='IoU')
    
    #print("Average Hausdorff Distance:", average_hausdorff_distance) # -used for initial bias assessment 
    #print("Dice Coefficient:", dice)
    #print("Jaccard Index:", jaccard)
    #print("Image index:", image_index)
    metrics_dice.append(dice)
    metrics_jaccard.append(jaccard)
    metrics_hausdorff.append(average_hausdorff_distance)
    image_number.append(image_index)
    if i < 6:  
        fig, ax = plt.subplots(ncols=3, figsize=(20, 20))
        ax[0].imshow(image, cmap ='gray')
        ax[0].set_title('Original')
        ax[1].imshow(shuffled_colormask)
        ax[1].set_title('Segmentation prediction') 
        ax[2].imshow(ground, cmap = plt.cm.nipy_spectral)
        ax[2].set_title('Ground Truth')
        plt.savefig(segmentation_results /f"Segmentation_Example{i+1}.png", dpi=300)
        plt.close()
        
    i = i + 1 

#----------------------------------
AHD = np.array(metrics_hausdorff)
DSI = np.array(metrics_dice)
IoU = np.array(metrics_jaccard)
index = np.array(image_number)
MeanAverage_Hausdorff_Distance =  np.mean(metrics_hausdorff)
Standard_Deviation_of_Hausdorff_Distance = np.std(metrics_hausdorff)
Average_Intersection_Over_Union = np.mean(metrics_jaccard)
Standard_Deviation_of_Intersection_Over_Union = np.std(metrics_jaccard)
Average_Dice_Similarity_Index = np.mean(metrics_dice)
Standard_Deviation_of_Dice_Similarity_Index = np.std(metrics_dice)
#-----------------------
df2 = pd.DataFrame(
    {
        "Image Number": index,
        "Average Hausdorff Distance": AHD,
        "Dice Coefficient": DSI,
        "Intersection over Union ": IoU,
        
    }
)
meanstd = pd.DataFrame(
    { 
        "" : pd.Categorical(["Average", "Standard Deviation"]),
        "Average Hausdorff Distance": np.array([MeanAverage_Hausdorff_Distance,  Standard_Deviation_of_Hausdorff_Distance]),
        "Intersection over Union": np.array([Average_Intersection_Over_Union,  Standard_Deviation_of_Intersection_Over_Union]),
        "Dice Coefficient" :np.array([Average_Dice_Similarity_Index,  Standard_Deviation_of_Dice_Similarity_Index]),
    }
)

df2.to_csv(segmentation_results /'BBBC039_segmentation_evaluation.csv', index = False)
meanstd.to_csv(segmentation_results /'BBBC039_SegmentationMetrics_mean,std.csv', index = False)
print("-------Evaluation Complete-----")
print(Path.cwd())

-------Evaluation Complete-----
C:\Users\Anant


Cell-Segmentation-with-scikit-image/
│
├── README.md
├── requirements.txt
├── LICENSE
│
├── data/
│   ├── sample_image.tif
│   └── sample_ground_truth.tif
│
├── notebooks/
│   └── segmentation_pipeline.ipynb
│
├── src/
│   ├── preprocessing.py
│   ├── segmentation.py
│   ├── evaluation.py
│   └── visualization.py
│
├── results/
│   ├── original.png
│   ├── segmentation.png
│   ├── ground_truth.png
│   ├── overlay.png
│   └── metrics.csv
│
└── figures/
    ├── pipeline.png
    ├── border_objects.png
    └── touching_nuclei.png